# D1 · Comparación inter-método (v3, 6 métodos)

**Spec:** [`docs/spec_D1_v3_codex_method_comparison.md`](../docs/spec_D1_v3_codex_method_comparison.md)  |  **Bloque:** D · Método  |  **Run de este set:** `ROXs42Bb_realigned`

Compara los 6 métodos de extracción (C2–C6) por pares y banda sobre 33 controles y emite `recommended_method` con el árbol v3 (nunca fija el canónico).

| | |
|---|---|
| **Entrada** | C2/C3/C4 + C5/C6 (sgf/lpm) + G1 verdicts + predictor Ec. 1 de C5 |
| **Salida (QC/productos)** | `stages/stage_x10_qc.json` |
| **Consume aguas abajo** | D2 (consume el canónico de config) |


## Qué hace D1 v3 y cómo decide

D1 compara los métodos **por pares y por banda** con un **t control-centrado** (df = n_controles − 1): la diferencia del objeto contra la distribución de las diferencias de los 33 controles — referenciado a controles por construcción. Bandas B1–B6 (continuo) y LHα/LHβ/LOI (líneas); umbrales congelados p<0.0455 (divergente) y p<0.0027 (fuerte).

**Novedades v3** ([`docs/spec_D1_v3_codex_method_comparison.md`](../docs/spec_D1_v3_codex_method_comparison.md)): set de 6 métodos (15 pares) con la familia espectral de Julo et al. 2025 (sgf/lpm), y árbol de recomendación congelado (preferencia psffit > lpm > psfsub > sgf > aperture > ls entre validados G1, con sgf excluido si el continuo de la compañera es ciencia o si el predictor Ec. 1 supera 0.10).

**Resultado 2026-07-15**: G1 validó {psffit, sgf, lpm} → **3 pares primarios** (psffit–sgf, psffit–lpm, sgf–lpm), todos con controles limpios. `optimal_psfsub` pasó a rejected: su T=0.667 histórico venía de un E4 anterior a la consolidación Psfao de C1; con el modelo vigente T=0.25 (< 0.4, no bias-bounded). **Veredicto = `divergent_continuum`**: B6 (rojo lejano) con t=+14.4/+13.6 de psffit contra sgf/lpm, mientras lpm–sgf solo difiere t=−3.1 — la familia espectral es ~consistente donde la espacial diverge (afila el diagnóstico hacia el residuo de halo rojo de psffit; salvedad: sgf/lpm comparten ŝ, sesgo común posible). LHα marginal (t≈−2.9).

`recommended_method = None` (regla: solo recomienda con `consistent`); elegibles registrados {psffit, lpm}. **El usuario mantuvo psffit como canónico** (2026-07-15, [`docs/d1_canonical_method_decision.md`](../docs/d1_canonical_method_decision.md)); B6 sigue como sistemática presupuestada.


## Cómo ejecutar de forma independiente

```bash
conda activate MUSE               # kernel/env con astropy + musepipe
export RUN=ROXs42Bb_realigned   # el run de este objeto
cd MUSE-accretion-pipeline                    # raíz del repo
bash scripts/stage_x10_compare.sh --run-id $RUN
```

Ligero (~6 s).

La celda de abajo hace lo mismo desde el notebook (guardada por `RUN`).


In [ ]:
import os, sys
# Localiza la raíz del repo ascendiendo hasta encontrar `musepipe/` (robusto a
# la profundidad: funciona con el cwd en notebooks/<obj>/, en notebooks/ o en la
# raíz). Añade la raíz (para `import musepipe`) y notebooks/ (para `_nbcommon`).
_d = os.getcwd()
while _d != os.path.dirname(_d):
    if os.path.isdir(os.path.join(_d, 'musepipe')) and os.path.isdir(os.path.join(_d, 'notebooks')):
        break
    _d = os.path.dirname(_d)
_root = _d
for _p in (_root, os.path.join(_root, 'notebooks')):
    if _p not in sys.path:
        sys.path.insert(0, _p)
import _nbcommon as nb
RUN_ID = nb.resolve_run_id('ROXs42Bb_realigned')
print('run  =', RUN_ID)
print('root =', _root)
print('dir  =', nb.run_dir(RUN_ID))
print('QC   =', nb.provenance_line('stages/stage_x10_qc.json', RUN_ID))


## Ejecutar o auditar


In [ ]:
RUN = False   # -> True para RE-EJECUTAR esta etapa (regenera su QC)

if RUN:
    cmd = 'bash scripts/stage_x10_compare.sh --run-id $RUN'.replace('$RUN', RUN_ID)
    print('ejecutando:', cmd)
    import subprocess
    subprocess.run(cmd, shell=True, cwd=str(nb.project_root()), check=True)
else:
    print('Modo auditoría (RUN=False): se carga el QC existente abajo.')


## QC / resultados


In [ ]:
qc = nb.load_qc_optional('stages/stage_x10_qc.json', RUN_ID)
nb.show(qc, keys=['verdict', 'action', 'recommended_method', 'primary_pairs', 'spec_version'], title='D1')


## Resultados que llevaron a la conclusión

Veredicto, pares primarios v3, t por banda de cada par primario, y caveats por método.


In [ ]:
if qc is None:
    print('(evidencia omitida: la etapa no se ha ejecutado para esta cadena)')
else:
    with nb.evidence_guard('D1', 'stages/stage_x10_qc.json'):
        from scipy.stats import t as t_dist
        q = nb.load_qc('stages/stage_x10_qc.json', RUN_ID)
        st = q['statistics']
        df = st['n_controls'] - 1
        T_DIV = t_dist.ppf(1 - st['p_divergent'] / 2, df)
        T_STR = t_dist.ppf(1 - st['p_strong'] / 2, df)
        print('spec:', q['spec_version'], '| veredicto:', q['verdict'], '| recommended:', q['recommended_method'])
        print('pares primarios:', q['primary_pairs'])
        print('elegibles (árbol v3):', q['recommendation_rules']['eligible'])
        print('caveat sgf:', q['method_caveats']['sgf']['excluded_from_recommendation'])
        for pp in q['primary_pairs']:
            print(f'\nt control-centrado ({pp}):')
            for b, t in q['t_matrix'][pp].items():
                if t is None: continue
                flag = '  <-- FUERTE' if abs(t) > T_STR else ('  <- marginal' if abs(t) > T_DIV else '')
                print(f'   {b:4s}: t = {t:+.2f}{flag}')


## Plot 1 — t control-centrado por par × banda (15 pares)

Del `t_matrix` del QC. Las filas primarias (psffit–sgf, psffit–lpm, sgf–lpm) gobiernan el veredicto; el patrón B6 (psffit vs familia espectral t≈+14, lpm–sgf ≈ −3) es la firma nueva que aporta la v3.


In [ ]:
try:
    import numpy as np
    import matplotlib.pyplot as plt
    from scipy.stats import t as t_dist
    q = nb.load_qc('stages/stage_x10_qc.json', RUN_ID)
    st = q['statistics']
    T_STR = t_dist.ppf(1 - st['p_strong'] / 2, st['n_controls'] - 1)
    tm = q['t_matrix']
    primary = list(q['primary_pairs'])
    pairs = primary + [p for p in tm if p not in primary]
    bands = list(tm[pairs[0]].keys())
    M = np.array([[np.nan if tm[p].get(b) is None else tm[p][b] for b in bands] for p in pairs])
    fig, ax = plt.subplots(figsize=(9.5, 0.42 * len(pairs) + 1.8))
    im = ax.imshow(M, cmap='RdBu_r', vmin=-6, vmax=6, aspect='auto')
    ax.set_xticks(range(len(bands))); ax.set_xticklabels(bands)
    labels = [('* ' if p in primary else '') + p.replace('_vs_', ' vs ') for p in pairs]
    ax.set_yticks(range(len(pairs))); ax.set_yticklabels(labels, fontsize=7)
    for i in range(len(pairs)):
        for j in range(len(bands)):
            v = M[i, j]
            if np.isfinite(v):
                ax.text(j, i, f'{v:.1f}', ha='center', va='center', fontsize=6, color='k' if abs(v) < 4 else 'w')
    ax.set_title(f'D1 v3 · t por par × banda (* = primario; |t|>{T_STR:.2f} fuerte)')
    fig.colorbar(im, label='t'); fig.tight_layout()
    outdir = nb.run_dir(RUN_ID) / 'plots' / 'd1_compare'; outdir.mkdir(parents=True, exist_ok=True)
    fig.savefig(outdir / 'tmatrix.png', dpi=110); print('figura ->', outdir / 'tmatrix.png'); plt.show()
except Exception as e:
    print('No se pudo generar el plot:', type(e).__name__, e)


## Plot 2 — los 3 pares primarios: qué driva `divergent_continuum`

t por banda de cada par primario con los umbrales divergente y fuerte. B6 con t≈+14 (psffit vs sgf/lpm) cruza holgadamente el umbral fuerte; entre sgf y lpm el continuo es casi consistente.


In [ ]:
try:
    import numpy as np
    import matplotlib.pyplot as plt
    from scipy.stats import t as t_dist
    q = nb.load_qc('stages/stage_x10_qc.json', RUN_ID)
    st = q['statistics']; df = st['n_controls'] - 1
    t_div = t_dist.ppf(1 - st['p_divergent'] / 2, df)
    t_str = t_dist.ppf(1 - st['p_strong'] / 2, df)
    primary = list(q['primary_pairs'])
    bands = list(q['t_matrix'][primary[0]].keys())
    x = np.arange(len(bands)); w = 0.8 / len(primary)
    fig, ax = plt.subplots(figsize=(10, 4.2))
    for k, pp in enumerate(primary):
        tv = [q['t_matrix'][pp].get(b) for b in bands]
        tv = [np.nan if v is None else v for v in tv]
        ax.bar(x + (k - (len(primary)-1)/2) * w, tv, width=w, label=pp.replace('_vs_', ' vs '))
    for s in (t_div, t_str):
        ax.axhline(s, color='k', ls=':', lw=0.8); ax.axhline(-s, color='k', ls=':', lw=0.8)
    ax.axhline(0, color='k', lw=0.6)
    ax.set_xticks(x); ax.set_xticklabels(bands); ax.set_ylabel('t control-centrado')
    ax.set_title('D1 v3 · pares primarios → divergent_continuum (B6 fuerte)')
    ax.legend(fontsize=8); fig.tight_layout()
    outdir = nb.run_dir(RUN_ID) / 'plots' / 'd1_compare'; outdir.mkdir(parents=True, exist_ok=True)
    fig.savefig(outdir / 'primary_pairs.png', dpi=110); print('figura ->', outdir / 'primary_pairs.png'); plt.show()
except Exception as e:
    print('No se pudo generar el plot:', type(e).__name__, e)


## Decisiones y notas
- **METHOD_ORDER de 6** con la familia espectral (Julo+25); pares primarios = validados por G1; protocolo anti cherry-picking intacto. · [`docs/spec_D1_v3_codex_method_comparison.md`](../docs/spec_D1_v3_codex_method_comparison.md)
- **optimal_psfsub → rejected**: su T histórico venía de un E4 pre-consolidación Psfao; verificado con worktree HEAD que el cambio no proviene del código nuevo.
- **B6 afilado**: psffit vs familia espectral t≈+14 con lpm–sgf ≈ −3; sigue como sistemática presupuestada (decisión 2026-07-10, heredada). · [`docs/d1_canonical_method_decision.md`](../docs/d1_canonical_method_decision.md)
- **El usuario mantuvo psffit** como canónico (2026-07-15); elegibles del árbol: {psffit, lpm}; sgf excluido de recomendación (continuo = ciencia). · [`docs/d1_canonical_method_decision.md`](../docs/d1_canonical_method_decision.md)


## Conclusión (registrada)

**D1 v3: `divergent_continuum` con 3 pares primarios limpios (psffit–sgf, psffit–lpm, sgf–lpm); canónico = psffit (decisión del usuario 2026-07-15).**

- **Fecha:** cadena D1 v3 sobre el run realineado (2026-07-15).
- **Estadístico:** t control-centrado, 33 controles (df=32), corr_length 11.44 canales (de G1).
- **Firma B6:** la familia espectral es ~consistente donde psffit diverge (t≈+14) → el residuo apunta al halo rojo del modelo espacial; salvedad de ŝ compartida entre sgf/lpm.
- **Downstream:** D2 calibró los 6 métodos (comparador de continuo = lpm, errata D2 v1.1); E1 re-confirmó la no-detección con 6 métodos; E3 con throughput E4 fresco.
- Todo provisional hasta cerrar el A-block.
